In [1]:
from pathlib import Path

files = sorted(Path("data").rglob("*.csv"))
for f in files:
    print(f.name, "|", round(f.stat().st_size / 1_000_000, 1), "MB")
print("Total files:", len(files))

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv | 77.1 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv | 76.9 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv | 58.3 MB
Monday-WorkingHours.pcap_ISCX.csv | 176.9 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv | 83.1 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv | 52.0 MB
Tuesday-WorkingHours.pcap_ISCX.csv | 135.1 MB
Wednesday-workingHours.pcap_ISCX.csv | 225.2 MB
Total files: 8


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

SAMPLE = 0.10   # keep 10% of each file so the notebook does not run out of memory
parts = []
for f in sorted(Path("data").rglob("*.csv")):
    chunks = pd.read_csv(f, chunksize=100_000, low_memory=False, encoding="latin-1")
    kept = []
    for c in chunks:
        c.columns = c.columns.str.strip()
        kept.append(c.sample(frac=SAMPLE, random_state=42))
    df_f = pd.concat(kept)
    df_f["source_file"] = f.name
    parts.append(df_f)
    print(f.name, "->", len(df_f), "rows kept")

data = pd.concat(parts, ignore_index=True)
print("Total rows:", len(data), "| columns:", data.shape[1])
print(data["Label"].value_counts())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv -> 22574 rows kept
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv -> 28647 rows kept
Friday-WorkingHours-Morning.pcap_ISCX.csv -> 19103 rows kept
Monday-WorkingHours.pcap_ISCX.csv -> 52992 rows kept
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> 28860 rows kept
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv -> 17037 rows kept
Tuesday-WorkingHours.pcap_ISCX.csv -> 44591 rows kept
Wednesday-workingHours.pcap_ISCX.csv -> 69270 rows kept
Total rows: 283074 | columns: 80
Label
BENIGN                          227436
DoS Hulk                         23098
PortScan                         15822
DDoS                             12753
DoS GoldenEye                     1064
FTP-Patator                        788
DoS slowloris                      574
DoS Slowhttptest                   568
SSH-Patator                        546
Bot                                195
Web Attack ï¿½ Brute Force         161
Web Attack ï¿½ 

In [3]:
# 1. Make sure every feature column is a number
label = data["Label"].astype(str).str.strip()
files = data["source_file"]
features = data.drop(columns=["Label", "source_file"]).apply(pd.to_numeric, errors="coerce")

# 2. Remove infinite and empty values
features = features.replace([np.inf, -np.inf], np.nan)
good = features.notna().all(axis=1)
print("Rows with empty or infinite values removed:", int((~good).sum()))
features, label, files = features[good], label[good], files[good]

# 3. Remove duplicate rows
dup = features.duplicated()
print("Duplicate rows removed:", int(dup.sum()))
features, label, files = features[~dup], label[~dup], files[~dup]

# 4. Make the answer column: 0 = normal (BENIGN), 1 = attack
y = (label.str.upper() != "BENIGN").astype(int)

# 5. Check the result
print("\nClean rows:", len(features), "| feature columns:", features.shape[1])
print("Normal (0):", int((y == 0).sum()), "| Attack (1):", int((y == 1).sum()))
print("Attack share:", round(y.mean() * 100, 1), "%")

Rows with empty or infinite values removed: 296
Duplicate rows removed: 13705

Clean rows: 269073 | feature columns: 78
Normal (0): 220522 | Attack (1): 48551
Attack share: 18.0 %


In [4]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

# 1. Split: 80% to learn from, 20% to test on (keeping the same attack share in both)
X_train, X_test, y_train, y_test = train_test_split(
    features, y, test_size=0.2, stratify=y, random_state=42)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

# 2. Train XGBoost
model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                      tree_method="hist", n_jobs=-1, random_state=42,
                      eval_metric="logloss")
model.fit(X_train, y_train)

# 3. Predict on the test rows (attack if the probability is 0.5 or more)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

# 4. Results
print("\n", classification_report(y_test, pred, target_names=["normal", "attack"], digits=4))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print("Normal passed correctly (TN):", tn)
print("False alarms (FP):", fp)
print("Attacks missed (FN):", fn)
print("Attacks caught (TP):", tp)
print("\nAttack recall:", round(tp / (tp + fn), 4))
print("False alarm rate:", round(fp / (fp + tn), 4))
print("PR-AUC:", round(average_precision_score(y_test, proba), 4))

Training rows: 215258 | Test rows: 53815

               precision    recall  f1-score   support

      normal     0.9995    0.9996    0.9995     44105
      attack     0.9980    0.9976    0.9978      9710

    accuracy                         0.9992     53815
   macro avg     0.9988    0.9986    0.9987     53815
weighted avg     0.9992    0.9992    0.9992     53815

Normal passed correctly (TN): 44086
False alarms (FP): 19
Attacks missed (FN): 23
Attacks caught (TP): 9687

Attack recall: 0.9976
False alarm rate: 0.0004
PR-AUC: 0.9999


In [5]:
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)

rows = []
for f in sorted(files.unique()):
    test_mask = (files == f).values
    n_all = int(test_mask.sum())
    yt = y[test_mask]
    n_attack = int(yt.sum())

    m = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                      tree_method="hist", n_jobs=-1, random_state=42,
                      eval_metric="logloss")
    m.fit(features[~test_mask], y[~test_mask])
    p = (m.predict_proba(features[test_mask])[:, 1] >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(yt, p, labels=[0, 1]).ravel()
    types = ", ".join(sorted(set(label[test_mask][yt == 1]))) if n_attack else "(none)"
    rows.append({
        "File left out": f[:32],
        "Rows": n_all,
        "Attack rows": n_attack,
        "Caught": int(tp),
        "Missed": int(fn),
        "False alarms": int(fp),
        "Recall": round(tp / (tp + fn), 3) if n_attack else None,
        "Attack types in this file": types,
    })
    print("done:", f)

result = pd.DataFrame(rows)
total_tp, total_fn = result["Caught"].sum(), result["Missed"].sum()
print("\nOverall recall on unseen days:", round(total_tp / (total_tp + total_fn), 3))
result

done: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
done: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
done: Friday-WorkingHours-Morning.pcap_ISCX.csv
done: Monday-WorkingHours.pcap_ISCX.csv
done: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
done: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
done: Tuesday-WorkingHours.pcap_ISCX.csv
done: Wednesday-workingHours.pcap_ISCX.csv

Overall recall on unseen days: 0.171


,File left out,Rows,Attack rows,Caught,Missed,False alarms,Recall,Attack types in this file
0,Friday-WorkingHours-Afternoon-DD,22515,12753,8064,4689,19,0.632,DDoS
1,Friday-WorkingHours-Afternoon-Po,27071,14467,47,14420,68,0.003,PortScan
2,Friday-WorkingHours-Morning.pcap,18734,194,0,194,9,0.000,Bot
3,Monday-WorkingHours.pcap_ISCX.cs,51790,0,0,0,24,NaN,(none)
4,Thursday-WorkingHours-Afternoon-,27321,1,0,1,5460,0.000,Infiltration
5,Thursday-WorkingHours-Morning-We,16446,227,3,224,5,0.013,"Web Attack ï¿½ Brute Force, Web Attack ï¿½ Sql Injection, Web Attack ï¿½ XSS"
6,Tuesday-WorkingHours.pcap_ISCX.c,43136,1045,2,1043,219,0.002,"FTP-Patator, SSH-Patator"
7,Wednesday-workingHours.pcap_ISCX,62060,19864,205,19659,15,0.010,"DoS GoldenEye, DoS Hulk, DoS Slowhttptest, DoS slowloris, Heartbleed"


In [6]:
import json
from pathlib import Path

Path("models").mkdir(exist_ok=True)

# 1. Which attack types are in each file?
t = pd.DataFrame({"file": files.str[:28].values, "label": label.values})
t = t[t["label"].str.upper() != "BENIGN"]
print(t.groupby(["file", "label"]).size().to_string())

# 2. Save the model from Step 6 and its column list
model.save_model("models/real_model.json")
with open("models/real_columns.json", "w") as f:
    json.dump(list(features.columns), f)

# 3. A demo file of real flows the model never trained on
rng = np.random.RandomState(7)
idx_n = rng.choice(X_test.index[(y_test == 0).values], 200, replace=False)
idx_a = rng.choice(X_test.index[(y_test == 1).values], 100, replace=False)
demo = X_test.loc[np.concatenate([idx_n, idx_a])].copy()
demo["Real label"] = label.loc[demo.index].values
demo = demo.sample(frac=1, random_state=1)
demo.to_csv("models/demo_flows.csv", index=False)

print("\nSaved:", sorted(p.name for p in Path("models").iterdir()))
print("Demo flows:", len(demo), "(200 normal + 100 attack)")

file                          label                       
Friday-WorkingHours-Afternoo  DDoS                            12753
                              PortScan                        14467
Friday-WorkingHours-Morning.  Bot                               194
Thursday-WorkingHours-Aftern  Infiltration                        1
Thursday-WorkingHours-Mornin  Web Attack ï¿½ Brute Force        160
                              Web Attack ï¿½ Sql Injection        4
                              Web Attack ï¿½ XSS                 63
Tuesday-WorkingHours.pcap_IS  FTP-Patator                       653
                              SSH-Patator                       392
Wednesday-workingHours.pcap_  DoS GoldenEye                    1064
                              DoS Hulk                        17678
                              DoS Slowhttptest                  556
                              DoS slowloris                     565
                              Heartbleed                 